# YouTube Content Performance Analysis

Which three YouTube accounts should the client test first?

This anonymized analysis compares format and topic performance using public creator data. Public channel identities are retained. The README sets out three proposed account tests; the client identity, specific business scope, and internal documents are omitted. Topic labels were AI-assisted from metadata and have not undergone full-video human review.


## 1. Load the public snapshot and define eligible samples


In [1]:
from pathlib import Path
try:
    ROOT = Path(__file__).resolve().parents[1]
except NameError:
    ROOT = Path.cwd().resolve()
    if not (ROOT / 'data').exists():
        ROOT = ROOT.parent
if not (ROOT / 'data').exists():
    raise FileNotFoundError('Run from the repository root or notebooks directory.')
OUT = ROOT / 'outputs' / 'tables'
OUT.mkdir(parents=True, exist_ok=True)
FIGURES = ROOT / 'outputs' / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)
if 'display' not in globals():
    try:
        from IPython.display import display
    except ImportError:
        def display(value):
            if hasattr(value, 'data'):
                value = value.data
            if hasattr(value, 'to_string'):
                print(value.to_string(index=False))
            else:
                print(value)

import pandas as pd
snapshot = pd.read_csv(ROOT / 'data/public_video_snapshot.csv', dtype={'video_id':'string','channel_id':'string'})
if snapshot.video_id.isna().any() or snapshot.video_id.duplicated().any():
    raise ValueError('Each snapshot row must have a unique video_id.')
format_df = snapshot.loc[snapshot.analysis_format_eligible.eq(True)].copy()
content_df = snapshot.loc[snapshot.analysis_topic_eligible.eq(True)].copy()
assert set(content_df.video_id).issubset(set(format_df.video_id))
print(f'Format analysis: {len(format_df):,} videos from {format_df.channel_id.nunique()} channels')
print(f'Topic analysis: {len(content_df):,} videos from {content_df.channel_id.nunique()} channels')
display(format_df[['channel_name','account_type','title','video_type','view_count','like_count','comment_count']].head())


Format analysis: 1,401 videos from 29 channels
Topic analysis: 1,341 videos from 29 channels


channel_name,account_type,title,video_type,view_count,like_count,comment_count
Attorney Ryan,Lawyer-led,“We Accept Your Resignation”,Shorts,10400748,394936.0,4065.0
Attorney Ryan,Lawyer-led,Is this a multi million dollar lawsuit??,Shorts,9858111,495504.0,9117.0
Attorney Ryan,Lawyer-led,I caught HR in a lie! #business #youtubehighfive #money,Shorts,4907487,253789.0,5529.0
Attorney Ryan,Lawyer-led,This is an ILLEGAL trick! #business #careeradvice,Shorts,3632203,144725.0,2441.0
Attorney Ryan,Lawyer-led,Woman sues golf course for $15 Million. Will she win?,Shorts,2962266,142509.0,5888.0


## 2. Calculate engagement rates without filling missing values


In [2]:
# 对两份数据执行相同处理
for df in [format_df, content_df]:

    # 确保指标是数值类型
    for col in ["view_count", "like_count", "comment_count"]:
        df[col] = pd.to_numeric(df[col], errors="raise")

    # 播放量为0时不计算互动率
    valid_views = df["view_count"].where(df["view_count"] > 0)

    df["engagement_rate"] = (
        df["like_count"] + df["comment_count"]
    ) / valid_views

# 检查格式比较数据中的Missing情况
print("Missing values in the format sample:")
print(
    format_df[
        ["view_count", "like_count", "comment_count", "engagement_rate"]
    ].isna().sum()
)

# 仅为展示创建百分比列，原始互动率仍保留为数值
preview = format_df[
    ["channel_name", "video_type", "view_count", "engagement_rate"]
].head().copy()

preview["engagement_rate"] = preview["engagement_rate"].map(
    lambda x: f"{x:.2%}" if pd.notna(x) else "Missing"
)

display(preview)

Missing values in the format sample:
view_count          0
like_count         51
comment_count       5
engagement_rate    56
dtype: int64


channel_name,video_type,view_count,engagement_rate
Attorney Ryan,Shorts,10400748,3.84%
Attorney Ryan,Shorts,9858111,5.12%
Attorney Ryan,Shorts,4907487,5.28%
Attorney Ryan,Shorts,3632203,4.05%
Attorney Ryan,Shorts,2962266,5.01%


## 3. Compare formats within each account group


In [3]:
# 按账户类型和Video format汇总
format_summary = (
    format_df
    .groupby(["account_type", "video_type"], observed=True)
    .agg(
        sample_videos=("video_id", "size"),
        sample_channels=("channel_id", "nunique"),
        median_views=("view_count", "median"),
        valid_engagement_videos=("engagement_rate", "count"),
        median_engagement_rate=("engagement_rate", "median")
    )
    .reset_index()
)

# 分别显示两组结果
for account_type in ["Lawyer-led", "AI-presented"]:

    print(f"\n{account_type}：Shorts vs Long-form")

    result = format_summary[
        format_summary["account_type"] == account_type
    ].copy()

    result = result.rename(columns={
        "video_type": "Video format",
        "sample_videos": "Videos",
        "sample_channels": "Channels",
        "median_views": "Median views",
        "valid_engagement_videos": "Valid engagement videos",
        "median_engagement_rate": "Median engagement"
    })

    # 只调整展示格式，不改变 format_summary 中的数值
    result["Median views"] = result["Median views"].map(
        lambda x: f"{x:,.0f}"
    )

    result["Median engagement"] = result["Median engagement"].map(
        lambda x: f"{x:.2%}" if pd.notna(x) else "Missing"
    )

    display(result.drop(columns="account_type"))


Lawyer-led：Shorts vs Long-form


Video format,Videos,Channels,Median views,Valid engagement videos,Median engagement
Long-form,672,16,"603,002",666,3.37%
Shorts,328,11,"7,115,579",324,4.14%



AI-presented：Shorts vs Long-form


Video format,Videos,Channels,Median views,Valid engagement videos,Median engagement
Long-form,71,2,"2,211",70,3.32%
Shorts,330,9,"256,624",285,2.11%


## 4. Compare Shorts and long-form within the same channel


In [4]:
# 每个Channel分别计算两种格式的表现
channel_format = (
    format_df
    .groupby(
        ["account_type", "channel_id", "channel_name", "video_type"],
        observed=True
    )
    .agg(
        video_count=("video_id", "size"),
        median_views=("view_count", "median"),
        valid_engagement=("engagement_rate", "count"),
        median_engagement=("engagement_rate", "median")
    )
    .reset_index()
)

# 将 Shorts 和长视频并排放置
channel_comparison = channel_format.pivot(
    index=["account_type", "channel_id", "channel_name"],
    columns="video_type",
    values=[
        "video_count",
        "median_views",
        "valid_engagement",
        "median_engagement"
    ]
)

channel_comparison.columns = [
    f"{metric}_{video_format}"
    for metric, video_format in channel_comparison.columns
]

channel_comparison = channel_comparison.reset_index()

# 只保留Channels with at least 5 videos in both formats
channel_comparison = channel_comparison[
    (channel_comparison["video_count_Shorts"] >= 5)
    & (channel_comparison["video_count_Long-form"] >= 5)
].copy()

# ShortsMedian views ÷ 长视频Median views
channel_comparison["views_ratio"] = (
    channel_comparison["median_views_Shorts"]
    / channel_comparison["median_views_Long-form"].where(
        channel_comparison["median_views_Long-form"] > 0
    )
)

# 互动率差值：单位为“百分点”
# 两种格式都至少有5条有效互动数据时才计算
valid_engagement = (
    (channel_comparison["valid_engagement_Shorts"] >= 5)
    & (channel_comparison["valid_engagement_Long-form"] >= 5)
)

channel_comparison["engagement_difference_pp"] = (
    (
        channel_comparison["median_engagement_Shorts"]
        - channel_comparison["median_engagement_Long-form"]
    ) * 100
).where(valid_engagement)

# 分别展示Lawyer-led和AI-presented
for account_type in ["Lawyer-led", "AI-presented"]:

    print(f"\n{account_type}：Within-channel comparison")

    result = channel_comparison[
        channel_comparison["account_type"] == account_type
    ][[
        "channel_name",
        "video_count_Shorts",
        "video_count_Long-form",
        "median_views_Shorts",
        "median_views_Long-form",
        "views_ratio",
        "median_engagement_Shorts",
        "median_engagement_Long-form",
        "engagement_difference_pp"
    ]].copy()

    result = result.rename(columns={
        "channel_name": "Channel",
        "video_count_Shorts": "Shorts videos",
        "video_count_Long-form": "Long-form videos",
        "median_views_Shorts": "Shorts median views",
        "median_views_Long-form": "Long-form median views",
        "views_ratio": "Shorts / Long-form median views",
        "median_engagement_Shorts": "Shorts median engagement",
        "median_engagement_Long-form": "Long-form median engagement",
        "engagement_difference_pp": "Engagement difference (pp)"
    })

    display(result)


Lawyer-led：Within-channel comparison


Channel,Shorts videos,Long-form videos,Shorts median views,Long-form median views,Shorts / Long-form median views,Shorts median engagement,Long-form median engagement,Engagement difference (pp)
Tom Grieve,11.0,39.0,506290.0,539301.0,0.938789,0.058549,0.039702,1.884702
"Tyler Chou, The Creators' Attorney",15.0,35.0,11103.0,16329.0,0.679956,0.039061,0.066326,-2.726520
The Civil Rights Lawyer,18.0,32.0,4623692.0,5401415.5,0.856015,0.047083,0.026821,2.026184
LegalEagle,14.0,36.0,5861773.5,5001472.0,1.172010,0.054947,0.027120,2.782714
AttorneyTom,19.0,31.0,989846.0,313062.0,3.161821,0.044279,0.040292,0.398652



AI-presented：Within-channel comparison


Channel,Shorts videos,Long-form videos,Shorts median views,Long-form median views,Shorts / Long-form median views,Shorts median engagement,Long-form median engagement,Engagement difference (pp)
Agente Davis,25.0,25.0,179497.0,142004.0,1.264028,0.012791,0.026916,-1.412543


## 5. Compare pooled content categories


In [5]:
# 统计每组两种格式的Videos量
format_share = (
    format_df
    .groupby(["account_type", "video_type"], observed=True)
    .size()
    .reset_index(name="video_count")
)

# 占比 = 该格式Videos ÷ 该账户组的样本视频总数
format_share["video_share"] = (
    format_share["video_count"]
    / format_share.groupby("account_type")["video_count"].transform("sum")
)

# 分别展示
for account_type in ["Lawyer-led", "AI-presented"]:

    print(f"\n{account_type}：长短视频占比")

    result = (
        format_share[
            format_share["account_type"] == account_type
        ]
        .drop(columns="account_type")
        .rename(columns={
            "video_type": "Video format",
            "video_count": "Videos",
            "video_share": "样本占比"
        })
        .reset_index(drop=True)
    )

    display(result)


Lawyer-led：长短视频占比


Video format,Videos,样本占比
Long-form,672,0.672
Shorts,328,0.328



AI-presented：长短视频占比


Video format,Videos,样本占比
Long-form,71,0.177057
Shorts,330,0.822943


## 6. Report topic sample coverage


In [6]:
# 按账户类型和Content category汇总
content_summary = (
    content_df
    .groupby(
        ["account_type", "content_category_label"],
        observed=True
    )
    .agg(
        sample_videos=("video_id", "size"),
        sample_channels=("channel_id", "nunique"),
        median_views=("view_count", "median"),
        valid_engagement_videos=("engagement_rate", "count"),
        median_engagement_rate=("engagement_rate", "median")
    )
    .reset_index()
)

# 在各账户组内部，按Median views排序
content_summary = content_summary.sort_values(
    ["account_type", "median_views"],
    ascending=[True, False]
)

for account_type in ["Lawyer-led", "AI-presented"]:

    print(f"\n{account_type}：Content category performance")

    result = content_summary[
        content_summary["account_type"] == account_type
    ].copy()

    # 提示样本局限，不删除这些类别
    result["Sample note"] = result.apply(
        lambda row: "；".join(
            message
            for condition, message in [
                (row["sample_videos"] < 5, "Fewer than 5 videos"),
                (row["sample_channels"] < 3, "Fewer than 3 channels"),
                (
                    row["valid_engagement_videos"] < 5,
                    "Fewer than 5 engagement observations"
                )
            ]
            if condition
        ),
        axis=1
    )

    result = result.rename(columns={
        "content_category_label": "Content category",
        "sample_videos": "Videos",
        "sample_channels": "Channels",
        "median_views": "Median views",
        "valid_engagement_videos": "Valid engagement videos",
        "median_engagement_rate": "Median engagement"
    })

    display(result.drop(columns="account_type").reset_index(drop=True))


Lawyer-led：Content category performance


Content category,Videos,Channels,Median views,Valid engagement videos,Median engagement,Sample note
Case stories,95,14,2524792.0,95,0.035667,
Entertainment and challenges,34,4,1178082.5,34,0.039493,
Courtroom coverage,64,4,996150.5,64,0.028318,
Legal education,323,15,981890.0,318,0.035037,
News and commentary,431,17,534284.0,426,0.039183,



AI-presented：Content category performance


Content category,Videos,Channels,Median views,Valid engagement videos,Median engagement,Sample note
Case stories,1,1,2282456.0,1,0.020938,Fewer than 5 videos；Fewer than 3 channels；Fewer than 5 engagement observations
Fictional legal stories,301,8,315329.0,256,0.021620,
News and commentary,10,1,122304.0,10,0.029262,Fewer than 3 channels
Legal education,82,2,6480.5,81,0.026916,Fewer than 3 channels


## 7. Separate topic comparisons by video format


In [7]:
# 按账户类型、Video format、Content category汇总
content_format_summary = (
    content_df
    .groupby(
        ["account_type", "video_type", "content_category_label"],
        observed=True
    )
    .agg(
        sample_videos=("video_id", "size"),
        sample_channels=("channel_id", "nunique"),
        median_views=("view_count", "median"),
        valid_engagement_videos=("engagement_rate", "count"),
        median_engagement_rate=("engagement_rate", "median")
    )
    .reset_index()
)

# 分别显示：律师Shorts、律师长视频、AI Shorts、AI长视频
for account_type in ["Lawyer-led", "AI-presented"]:
    for video_type in ["Shorts", "Long-form"]:

        result = content_format_summary[
            (content_format_summary["account_type"] == account_type)
            & (content_format_summary["video_type"] == video_type)
        ].copy()

        if result.empty:
            continue

        print(f"\n{account_type}｜{video_type}：Content category performance")

        result = result.sort_values(
            "median_views",
            ascending=False
        )

        result["Sample note"] = result.apply(
            lambda row: "；".join(
                message
                for condition, message in [
                    (row["sample_videos"] < 5, "Fewer than 5 videos"),
                    (row["sample_channels"] < 3, "Fewer than 3 channels"),
                    (
                        row["valid_engagement_videos"] < 5,
                        "Fewer than 5 engagement observations"
                    )
                ]
                if condition
            ),
            axis=1
        )

        result = result.rename(columns={
            "content_category_label": "Content category",
            "sample_videos": "Videos",
            "sample_channels": "Channels",
            "median_views": "Median views",
            "valid_engagement_videos": "Valid engagement videos",
            "median_engagement_rate": "Median engagement"
        })

        display(result.drop(columns=["account_type","video_type"]).reset_index(drop=True))


Lawyer-led｜Shorts：Content category performance


Content category,Videos,Channels,Median views,Valid engagement videos,Median engagement,Sample note
Entertainment and challenges,20,3,82701307.0,20,0.030374,
Legal education,145,8,12079075.0,143,0.041746,
Case stories,36,5,4300543.5,36,0.044420,
News and commentary,81,8,296694.0,79,0.041092,



Lawyer-led｜Long-form：Content category performance


Content category,Videos,Channels,Median views,Valid engagement videos,Median engagement,Sample note
Case stories,59,11,1392153.0,59,0.031352,
Courtroom coverage,64,4,996150.5,64,0.028318,
Entertainment and challenges,14,3,989579.5,14,0.041256,
News and commentary,350,14,583528.5,347,0.038859,
Legal education,178,10,182805.5,175,0.027314,



AI-presented｜Shorts：Content category performance


Content category,Videos,Channels,Median views,Valid engagement videos,Median engagement,Sample note
Case stories,1,1,2282456.0,1,0.020938,Fewer than 5 videos；Fewer than 3 channels；Fewer than 5 engagement observations
Fictional legal stories,301,8,315329.0,256,0.021620,
Legal education,17,1,179497.0,17,0.011588,Fewer than 3 channels
News and commentary,4,1,170499.0,4,0.012797,Fewer than 5 videos；Fewer than 3 channels；Fewer than 5 engagement observations



AI-presented｜Long-form：Content category performance


Content category,Videos,Channels,Median views,Valid engagement videos,Median engagement,Sample note
News and commentary,6,1,100765.0,6,0.035148,Fewer than 3 channels
Legal education,65,2,1689.0,64,0.032548,Fewer than 3 channels


## 8. Check creator concentration in lawyer-led Shorts


In [8]:
# 先检查Lawyer-led Shorts 的Channel composition
lawyer_shorts = content_df[
    (content_df["account_type"] == "Lawyer-led")
    & (content_df["video_type"] == "Shorts")
].copy()

channel_contribution = (
    lawyer_shorts
    .groupby(
        ["content_category_label", "channel_id", "channel_name"],
        observed=True
    )
    .agg(
        sample_videos=("video_id", "size"),
        median_views=("view_count", "median"),
        valid_engagement_videos=("engagement_rate", "count"),
        median_engagement_rate=("engagement_rate", "median")
    )
    .reset_index()
)

# 每个Channel占该类别样本视频的比例
category_total = (
    channel_contribution
    .groupby("content_category_label")["sample_videos"]
    .transform("sum")
)

channel_contribution["sample_share"] = (
    channel_contribution["sample_videos"] / category_total
)

# 每个类别单独展示
category_order = [
    "Entertainment and challenges",
    "Legal education",
    "Case stories",
    "News and commentary"
]

for category in category_order:

    print(f"\nLawyer-led Shorts｜{category}：Channel composition")

    result = channel_contribution[
        channel_contribution["content_category_label"] == category
    ].sort_values(
        ["sample_videos", "median_views"],
        ascending=False
    )

    result = result[[
        "channel_name",
        "sample_videos",
        "sample_share",
        "median_views",
        "valid_engagement_videos",
        "median_engagement_rate"
    ]].rename(columns={
        "channel_name": "Channel",
        "sample_videos": "Videos",
        "sample_share": "Share of category videos",
        "median_views": "Median views",
        "valid_engagement_videos": "Valid engagement videos",
        "median_engagement_rate": "Median engagement"
    })

    display(result.reset_index(drop=True))


Lawyer-led Shorts｜Entertainment and challenges：Channel composition


Channel,Videos,Share of category videos,Median views,Valid engagement videos,Median engagement
Law By Mike,11,0.55,107108872.0,11,0.023465
AttorneyTom,8,0.40,574254.0,8,0.048025
"Tyler Chou, The Creators' Attorney",1,0.05,9323.0,1,0.031857



Lawyer-led Shorts｜Legal education：Channel composition


Channel,Videos,Share of category videos,Median views,Valid engagement videos,Median engagement
Ugo Lord: The Modern Attorney,50,0.344828,14396077.0,49,0.041746
Attorney Ryan,30,0.206897,1166304.5,30,0.048411
Law By Mike,29,0.200000,87909102.0,29,0.036458
Mike Rafi - Personal Injury Lawyer,14,0.096552,9301006.0,13,0.040561
AttorneyTom,10,0.068966,2046454.5,10,0.041539
Charles Buist - Personal Injury Lawyer,6,0.041379,57861.0,6,0.032829
Tom Grieve,3,0.020690,3324035.0,3,0.025292
"Tyler Chou, The Creators' Attorney",3,0.020690,18392.0,3,0.038223



Lawyer-led Shorts｜Case stories：Channel composition


Channel,Videos,Share of category videos,Median views,Valid engagement videos,Median engagement
The Civil Rights Lawyer,14,0.388889,4618782.0,14,0.049566
Attorney Ryan,8,0.222222,1047187.5,8,0.044400
Mike Rafi - Personal Injury Lawyer,7,0.194444,11383823.0,7,0.049187
Charles Buist - Personal Injury Lawyer,5,0.138889,221730.0,5,0.039476
Law By Mike,2,0.055556,113310393.0,2,0.023445



Lawyer-led Shorts｜News and commentary：Channel composition


Channel,Videos,Share of category videos,Median views,Valid engagement videos,Median engagement
Charles Buist - Personal Injury Lawyer,38,0.469136,81324.0,36,0.029834
LegalEagle,12,0.148148,6060152.0,12,0.053701
"Tyler Chou, The Creators' Attorney",11,0.135802,11103.0,11,0.044763
Tom Grieve,8,0.098765,506118.5,8,0.067913
Mike Rafi - Personal Injury Lawyer,4,0.049383,8260280.5,4,0.051511
The Civil Rights Lawyer,4,0.049383,4916740.0,4,0.040320
CLR Bruce Rivers,2,0.024691,1488406.0,2,0.040180
Attorney Ryan,2,0.024691,793382.0,2,0.049453


## 9. Recompute topic medians without a dominant creator


In [9]:
# 两种Scenario：All channels、暂时Exclude Law By Mike
comparison_data = {
    "All channels": lawyer_shorts,
    "Exclude Law By Mike": lawyer_shorts[
        lawyer_shorts["channel_name"] != "Law By Mike"
    ]
}

comparison_tables = []

for scenario, df in comparison_data.items():

    summary = (
        df.groupby("content_category_label", observed=True)
        .agg(
            sample_videos=("video_id", "size"),
            sample_channels=("channel_id", "nunique"),
            median_views=("view_count", "median"),
            valid_engagement_videos=("engagement_rate", "count"),
            median_engagement_rate=("engagement_rate", "median")
        )
        .reset_index()
    )

    summary["scenario"] = scenario
    comparison_tables.append(summary)

sensitivity_summary = pd.concat(
    comparison_tables,
    ignore_index=True
)

# 每个类别并排查看两种Scenario
for category in category_order:

    print(f"\nLawyer-led Shorts｜{category}")

    result = sensitivity_summary[
        sensitivity_summary["content_category_label"] == category
    ][[
        "scenario",
        "sample_videos",
        "sample_channels",
        "median_views",
        "valid_engagement_videos",
        "median_engagement_rate"
    ]].rename(columns={
        "scenario": "Scenario",
        "sample_videos": "Videos",
        "sample_channels": "Channels",
        "median_views": "Median views",
        "valid_engagement_videos": "Valid engagement videos",
        "median_engagement_rate": "Median engagement"
    })

    display(result.reset_index(drop=True))


Lawyer-led Shorts｜Entertainment and challenges


Scenario,Videos,Channels,Median views,Valid engagement videos,Median engagement
All channels,20,3,82701307.0,20,0.030374
Exclude Law By Mike,9,2,446142.0,9,0.047558



Lawyer-led Shorts｜Legal education


Scenario,Videos,Channels,Median views,Valid engagement videos,Median engagement
All channels,145,8,12079075.0,143,0.041746
Exclude Law By Mike,116,7,9609944.5,114,0.043313



Lawyer-led Shorts｜Case stories


Scenario,Videos,Channels,Median views,Valid engagement videos,Median engagement
All channels,36,5,4300543.5,36,0.044420
Exclude Law By Mike,34,4,4142003.0,34,0.044836



Lawyer-led Shorts｜News and commentary


Scenario,Videos,Channels,Median views,Valid engagement videos,Median engagement
All channels,81,8,296694.0,79,0.041092
Exclude Law By Mike,81,8,296694.0,79,0.041092


## 10. Compare topics against each channel Shorts baseline


In [10]:
MIN_SAMPLE = 5

# 1. 每个Channel自己的 Shorts 基准
# 使用 format_df，避免漏掉题材不明或非法律主题的 Shorts
shorts_baseline = (
    format_df[
        format_df["video_type"] == "Shorts"
    ]
    .groupby(
        ["account_type", "channel_id"],
        observed=True
    )
    .agg(
        baseline_videos=("video_id", "size"),
        baseline_median_views=("view_count", "median"),
        baseline_valid_engagement=("engagement_rate", "count"),
        baseline_median_engagement=("engagement_rate", "median")
    )
    .reset_index()
)

# 2. 每个Channel、每种题材的 Shorts 表现
shorts_categories = (
    content_df[
        content_df["video_type"] == "Shorts"
    ]
    .groupby(
        [
            "account_type",
            "channel_id",
            "channel_name",
            "content_category_label"
        ],
        observed=True
    )
    .agg(
        category_videos=("video_id", "size"),
        category_valid_views=("view_count", "count"),
        category_median_views=("view_count", "median"),
        category_valid_engagement=("engagement_rate", "count"),
        category_median_engagement=("engagement_rate", "median")
    )
    .reset_index()
)

# 3. 合并同一Channel的基准
relative_shorts = shorts_categories.merge(
    shorts_baseline,
    on=["account_type", "channel_id"],
    how="left",
    validate="many_to_one"
)

# 4. 播放量相对倍数
# 例如 1.50 表示：该题材播放中位数是Channel基准的1.5倍
relative_shorts["relative_views"] = (
    relative_shorts["category_median_views"]
    / relative_shorts["baseline_median_views"].where(
        relative_shorts["baseline_median_views"] > 0
    )
).where(
    relative_shorts["category_valid_views"] >= MIN_SAMPLE
)

# 5. 互动率相对于Channel基准的差值，单位为百分点
valid_engagement_comparison = (
    (relative_shorts["category_valid_engagement"] >= MIN_SAMPLE)
    & (relative_shorts["baseline_valid_engagement"] >= MIN_SAMPLE)
)

relative_shorts["engagement_difference_pp"] = (
    (
        relative_shorts["category_median_engagement"]
        - relative_shorts["baseline_median_engagement"]
    ) * 100
).where(valid_engagement_comparison)

# 6. Sample note
relative_shorts["sample_note"] = relative_shorts.apply(
    lambda row: "；".join(
        message
        for condition, message in [
            (
                row["category_valid_views"] < MIN_SAMPLE,
                "Fewer than 5 view observations"
            ),
            (
                row["category_valid_engagement"] < MIN_SAMPLE,
                "Fewer than 5 engagement observations"
            )
        ]
        if condition
    ),
    axis=1
)

# 7. Lawyer-led、AI-presented分别展示
for account_type in ["Lawyer-led", "AI-presented"]:

    print(f"\n{account_type}：Categories relative to the same channel Shorts baseline")

    result = relative_shorts[
        relative_shorts["account_type"] == account_type
    ].sort_values(
        ["channel_name", "relative_views"],
        ascending=[True, False],
        na_position="last"
    )

    result = result[[
        "channel_name",
        "content_category_label",
        "baseline_videos",
        "category_videos",
        "baseline_median_views",
        "category_median_views",
        "relative_views",
        "category_valid_engagement",
        "baseline_median_engagement",
        "category_median_engagement",
        "engagement_difference_pp",
        "sample_note"
    ]].rename(columns={
        "channel_name": "Channel",
        "content_category_label": "Content category",
        "baseline_videos": "Channel Shorts sample",
        "category_videos": "Category videos",
        "baseline_median_views": "Channel median views",
        "category_median_views": "Category median views",
        "relative_views": "Relative views",
        "category_valid_engagement": "Valid category engagement",
        "baseline_median_engagement": "Channel median engagement",
        "category_median_engagement": "Category median engagement",
        "engagement_difference_pp": "Engagement difference (pp)",
        "sample_note": "Sample note"
    })

    display(result.reset_index(drop=True))


Lawyer-led：Categories relative to the same channel Shorts baseline


Channel,Content category,Channel Shorts sample,Category videos,Channel median views,Category median views,Relative views,Valid category engagement,Channel median engagement,Category median engagement,Engagement difference (pp),Sample note
Attorney Ryan,Legal education,50,30,1188204.5,1166304.5,0.981569,30,0.048310,0.048411,0.010105,
Attorney Ryan,Case stories,50,8,1188204.5,1047187.5,0.881319,8,0.048310,0.044400,-0.390972,
Attorney Ryan,News and commentary,50,2,1188204.5,793382.0,NaN,2,0.048310,0.049453,NaN,Fewer than 5 view observations；Fewer than 5 engagement observations
AttorneyTom,Legal education,19,10,989846.0,2046454.5,2.067447,10,0.044279,0.041539,-0.274022,
AttorneyTom,Entertainment and challenges,19,8,989846.0,574254.0,0.580145,8,0.044279,0.048025,0.374558,
CLR Bruce Rivers,News and commentary,2,2,1488406.0,1488406.0,NaN,2,0.040180,0.040180,NaN,Fewer than 5 view observations；Fewer than 5 engagement observations
Charles Buist - Personal Injury Lawyer,Case stories,49,5,81007.0,221730.0,2.737171,5,0.031701,0.039476,0.777473,
Charles Buist - Personal Injury Lawyer,News and commentary,49,38,81007.0,81324.0,1.003913,36,0.031701,0.029834,-0.186666,
Charles Buist - Personal Injury Lawyer,Legal education,49,6,81007.0,57861.0,0.714272,6,0.031701,0.032829,0.112765,
Law By Mike,Entertainment and challenges,50,11,94385992.5,107108872.0,1.134796,11,0.028952,0.023465,-0.548700,



AI-presented：Categories relative to the same channel Shorts baseline


Channel,Content category,Channel Shorts sample,Category videos,Channel median views,Category median views,Relative views,Valid category engagement,Channel median engagement,Category median engagement,Engagement difference (pp),Sample note
Agente Davis,Legal education,25,17,179497.0,179497.0,1.000000,17,0.012791,0.011588,-0.120307,
Agente Davis,Case stories,25,1,179497.0,2282456.0,NaN,1,0.012791,0.020938,NaN,Fewer than 5 view observations；Fewer than 5 engagement observations
Agente Davis,Fictional legal stories,25,3,179497.0,129687.0,NaN,3,0.012791,0.014718,NaN,Fewer than 5 view observations；Fewer than 5 engagement observations
Agente Davis,News and commentary,25,4,179497.0,170499.0,NaN,4,0.012791,0.012797,NaN,Fewer than 5 view observations；Fewer than 5 engagement observations
Court Verdict,Fictional legal stories,50,50,63045.5,63045.5,1.000000,50,0.026327,0.026327,0.000000,
CourtOnCam,Fictional legal stories,50,47,27623.5,27764.0,1.005086,47,0.010730,0.010769,0.003901,
Justice Stories AI,Fictional legal stories,50,50,246861.5,246861.5,1.000000,5,0.019965,0.019965,0.000000,
Law Dose,Fictional legal stories,50,50,95870.0,95870.0,1.000000,50,0.011402,0.011402,0.000000,
Worldwide Court,Fictional legal stories,50,50,1213309.0,1213309.0,1.000000,50,0.024657,0.024657,0.000000,
Your Verdict HQ,Fictional legal stories,50,50,637492.0,637492.0,1.000000,50,0.024163,0.024163,0.000000,


## 11. Export tables and create figures


In [11]:
import sys
sys.path.insert(0,str(ROOT/'src'))
from plot_svg import bars, COLORS
tables={'format_by_group':format_summary,'format_by_channel':channel_format,
        'within_channel_format_comparison':channel_comparison,'content_by_group':content_summary,
        'content_by_format':content_format_summary,'lawyer_shorts_channel_contribution':channel_contribution,
        'lawyer_shorts_sensitivity':sensitivity_summary,'shorts_relative_to_channel_baseline':relative_shorts}
for name,table in tables.items():table.to_csv(OUT/f'{name}.csv',index=False)
quality=pd.DataFrame([{'sample':'raw snapshot','videos':len(snapshot),'channels':snapshot.channel_id.nunique()},
                      {'sample':'format analysis','videos':len(format_df),'channels':format_df.channel_id.nunique()},
                      {'sample':'topic analysis','videos':len(content_df),'channels':content_df.channel_id.nunique()}])
quality.to_csv(OUT/'sample_flow.csv',index=False)
bars(FIGURES/'format_median_views.svg',[(f'{r.account_type} | {r.video_type}',r.median_views,COLORS[0 if r.account_type=='Lawyer-led' else 1]) for r in format_summary.itertuples(index=False)],
     'Video format comparisons depend on the channel group','Historical high-view sample; medians describe these selected videos','Median cumulative public views')

law=content_format_summary[(content_format_summary.account_type=='Lawyer-led') & (content_format_summary.video_type=='Shorts')].sort_values('median_views',ascending=False)
bars(FIGURES/'lawyer_shorts_topics.svg',[(f'{r.content_category_label} (n={r.sample_videos})',r.median_views,COLORS[0]) for r in law.itertuples(index=False)],
     'Lawyer-led Shorts: category performance needs a channel check','AI-assisted topic labels; not yet validated by full-video human review','Median cumulative public views')
sensitivity_rows=[]
for category in ['Entertainment and challenges','Legal education','Case stories']:
    scenarios=sensitivity_summary[sensitivity_summary.content_category_label.eq(category)]
    for r in scenarios.itertuples(index=False):
        sensitivity_rows.append((f'{r.content_category_label} | {r.scenario}',r.median_views,COLORS[0 if r.scenario=='All channels' else 1]))
bars(FIGURES/'channel_sensitivity.svg',sensitivity_rows,
     'One creator can change the pooled topic ranking','Lawyer-led Shorts; compare all channels with the sample excluding Law By Mike','Median cumulative public views')
display(quality)


sample,videos,channels
raw snapshot,1407,30
format analysis,1401,29
topic analysis,1341,29
